# V0.8 Multi-Agent Boundary Lab

问题：两个 Agent 如何通信、授权和共享资源而不越权？

这个 notebook 逐格展示 multi-agent 中 identity、IPC、capability delegation 和 resource sharing 的边界。

## Mode

`deterministic`：Model decision = SCRIPTED，Kernel execution = REAL。

`real_model`：Model decision = REAL OpenAI-compatible，Kernel execution = REAL。优先使用 `AGENTKERNEL_LAB_LLM_*`，否则复用仓库本地 `.minicode/config.json`。不会展示 hidden chain-of-thought，也不会展示 API key。

In [ ]:
MODE = "deterministic"

from pathlib import Path
import sys

def find_agentkernel_root(start: Path) -> Path:
    for path in (start, *start.parents):
        if (path / "agentkernel").is_dir() and (path / "labs").is_dir():
            return path
    raise RuntimeError("Run this notebook from the AgentKernel repo root or the labs directory.")

REPO_ROOT = find_agentkernel_root(Path.cwd().resolve())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from labs import create_lab

lab = create_lab("v08", mode=MODE)


## Step 1: Setup

创建 parent/child Agent、parent/child Process、IPC 资源和一个 parent-owned artifact。

In [ ]:
_ = lab.setup()

## Step 2: Inspect child model request

delegation 前 child 看不到 parent 的 tool authority。

In [ ]:
_ = lab.show_model_request()

## Step 3: Ask deterministic or real model

模型可以解释需要哪些权限，但不能自己授予权限。

In [ ]:
_ = lab.model_step()

## Step 4: Child before delegation

child 尝试使用 parent tool，会被 Kernel 拒绝。

In [ ]:
_ = lab.child_before_delegation()

## Step 5: Delegate and execute

parent 委托一个缩小的 capability 后，child 才能执行相应 tool。

In [ ]:
_ = lab.delegate_and_execute()

## Step 6: IPC reference is not permission

IPC 可以传递 ResourceHandle URI，但 URI 本身不是权限。

In [ ]:
_ = lab.ipc_resource_reference()

## Step 7: Explicit share and read

显式 resource share 后，child 才能通过 ResourceService 读取。

In [ ]:
_ = lab.share_and_read()

## Summary

这个实验回答：通信、授权、资源共享是三个不同 Kernel 机制。

In [ ]:
_ = lab.summary()
_ = lab.close()